# Cleopatra weather styles on **real** data

The ECMWF-derived `DATA_STYLES` weather presets, applied to genuine gridded fields:

- **temperature** styles on real **ERA5 2 m air temperature** (Europe), and
- **precipitation** styles on a real **NOAH land-model precipitation** raster (read with pyramids), plus
- **flow accumulation** on a real hydrology raster.

Each preset is applied in one call with `ArrayGlyph(field, extent=..., ax=ax).plot(style=style, ...)`.

In [ ]:
%matplotlib inline
import os, sys
_wt = r"C:/python-environments/worktrees/cleopatra/perceptual-palettes/src"
if os.path.isdir(_wt) and _wt not in sys.path:
    sys.path.insert(0, _wt)
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from pyramids.dataset import Dataset
import cleopatra
from cleopatra.array_glyph import ArrayGlyph
print("cleopatra", cleopatra.__version__)

DATA = Path("../data")

def read_raster(path, band=0):
    ds = Dataset.read_file(str(path))
    a = np.asarray(ds.read_array(), dtype=float)
    if a.ndim == 3: a = a[band]
    nod = ds.no_data_value[0] if ds.no_data_value else None
    if nod is not None and np.isfinite(nod): a = np.where(np.isclose(a, nod), np.nan, a)
    r, c = a.shape; x0, dx, _, y0, _, dy = ds.geotransform
    return a, [x0, x0 + c * dx, y0 + r * dy, y0]

# real ERA5 2 m temperature (deg C), pick the most dynamic day
z = np.load(DATA / "europe_t2m.npz", allow_pickle=True)
cel = z["celsius"]
day = int(np.argmax([np.nanmax(f) - np.nanmin(f) for f in cel]))
t2m = cel[day].astype(float); t2m_ext = [float(v) for v in z["extent"]]
# real precipitation + flow accumulation via pyramids (GDAL)
precip, precip_ext = read_raster(DATA / "gis/noah-precipitation-1979-europe.tif")
facc, facc_ext = read_raster(DATA / "gis/acc4000.tif")
print("t2m:", t2m.shape, "degC", (round(np.nanmin(t2m),1), round(np.nanmax(t2m),1)),
      "| precip:", precip.shape, (round(np.nanmin(precip),1), round(np.nanmax(precip),1)),
      "| flow_acc:", facc.shape)

def gallery(styles, field, extent, title, ncol=3):
    styles = [s for s in styles]
    nrow = int(np.ceil(len(styles) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(13, 3.3 * nrow))
    gext = [extent[0], extent[2], extent[1], extent[3]]   # [xmin, ymin, xmax, ymax]
    for ax, style in zip(np.atleast_1d(axes).ravel(), styles):
        ArrayGlyph(field, extent=gext, ax=ax).plot(style=style, add_colorbar=False,
                                                   title=style, title_size=9)
        ax.set_xticks([]); ax.set_yticks([])
    for ax in np.atleast_1d(axes).ravel()[len(styles):]:
        ax.set_visible(False)
    fig.suptitle(title, fontweight="bold"); fig.tight_layout(); plt.show()

## Temperature styles — real ERA5 2 m temperature (°C, Europe)

These presets assume °C, which matches ERA5. The two `temperature_flame*` styles tie opacity to value
(hot glows, cool fades) — the CAMS aerosol technique recoloured for heat.

In [ ]:
temp_styles = ["temperature_2m", "air_temperature", "mean_temperature_2m",
               "min_temperature_2m", "max_temperature_2m", "dewpoint_temperature_2m",
               "potential_temperature", "temperature", "temperature_flame", "temperature_flame_amber"]
gallery(temp_styles, t2m, t2m_ext, "Temperature styles on real ERA5 2 m temperature")

## Precipitation styles — real NOAH precipitation

The precipitation family (total / convective / large-scale precip, rain & snowfall rates) on a real
precipitation field.

In [ ]:
precip_styles = ["total_precipitation", "precipitation", "convective_precipitation",
                 "large_scale_precipitation", "precipitation_rate", "convective_rainfall_rate",
                 "large_scale_rainfall_rate", "rainfall", "snowfall", "snowfall_rate_water_equivalent"]
gallery(precip_styles, precip, precip_ext, "Precipitation styles on real NOAH precipitation")

## Flow accumulation — real hydrology raster

`flow_accumulation` uses a symmetric-log norm with value-linked opacity, so the channel network stands out
against faded low-accumulation cells.

In [ ]:
gallery(["flow_accumulation"], facc, facc_ext, "flow_accumulation on a real flow-accumulation raster", ncol=1)